In [ ]:
from importlib.util import find_spec
if find_spec("matplotlib") is None or find_spec("scipy") is None or find_spec("numpy") is None \
    or find_spec("pandas") is None or find_spec("seaborn") is None or find_spec("pymoo") is None:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib", "scipy", "numpy", "pandas", "seaborn", "pymoo"])

import warnings
import json 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import pymoo
import time

from designTool.constants import gravity
from designTool.analyze import analyze
from designTool.geometry import geometry
from designTool.standard_airplane import standard_airplane
from designTool.plots import plot_geometry
from designTool.auxiliary import atmosphere
from designTool.aerodynamics import aerodynamics

np.random.seed(1234)
pd.options.display.float_format = "{:,.3f}".format
warnings.simplefilter("error")

TOMAV_KWARGS = ["delta_xr_w", "x_tank_c_w"]
CONDICAO_SEGUNDO_SEGMENTO = {
    "Mach": 0.30,
    "altitude": 0,
    "n_engines_failed": 1,
    "highlift_config": "takeoff",
    "lg_down": 1,
    "h_ground": 10.668,
}

In [ ]:
def aerodynamics_condicao_e_CL(airplane: dict, condicao: dict, CL: float) -> tuple[float, float, dict]:
    CD, CLmax, dragDict = aerodynamics(airplane, condicao["Mach"], condicao["altitude"], CL,
                                       n_engines_failed=condicao["n_engines_failed"],
                                       highlift_config=condicao["highlift_config"], lg_down=condicao["lg_down"],
                                       h_ground=condicao["h_ground"])
    return CD, CLmax, dragDict

def criar_airplane_parametros(airplane_name: str, parametros: dict) -> dict:
    airplane = standard_airplane(airplane_name, **{parametro:valor for parametro,valor in \
            parametros.items() if airplane_name=="Tomav" and parametro in TOMAV_KWARGS})

    for parametro, valor in parametros.items():
        if airplane_name=="Tomav" and parametro in TOMAV_KWARGS:
            pass
        airplane["inputs"][parametro] = valor

    return airplane

def calcular_restricoes(airplane: dict) -> dict | None:
    try:
        analyze(airplane)
        W0 = airplane["thrust_matching"]["W0"]
        T0 = airplane["thrust_matching"]["T0"]
        sigma = atmosphere(airplane["inputs"]["altitude_takeoff"], airplane["inputs"]["deltaISA_takeoff"])["density"] / 1.225
        _, CLmax_TO, _ = aerodynamics_condicao_e_CL(airplane, CONDICAO_SEGUNDO_SEGMENTO, 0)
        d_TO = 0.2387 / (sigma * CLmax_TO * airplane["inputs"]["S_w"]) * W0**2 / T0
    except Exception:
        return None
    if d_TO is None:
        return None
    
    restricoes = {
        "deltaS_wlan": airplane["thrust_matching"]["deltaS_wlan"], # 1
        "SM_fwd": airplane["balance"]["SM_fwd"], # 2
        "SM_aft": airplane["balance"]["SM_aft"], # 3
        "frac_nlg_aft": airplane["landing_gear"]["frac_nlg_aft"], # 4
        "frac_nlg_fwd": airplane["landing_gear"]["frac_nlg_fwd"], # 5
        "alpha_tipback": airplane["landing_gear"]["alpha_tipback"], # 6
        "alpha_tailstrike": airplane["landing_gear"]["alpha_tailstrike"], # 7
        "phi_overturn": airplane["landing_gear"]["phi_overturn"], # 8
        # Restrição 9 já está definida no input b_tank_b_w = 0.95
        "b_w": airplane["geometry"]["b_w"], # 10
        # Restrição 11 já está definida no input y_mlg = 5.5
        "tail_height": airplane["geometry"]["zt_v"] - airplane["inputs"]["z_lg"], # 12
        # Restrições adicionais:
        "CLv": airplane["balance"]["CLv"], # 13
        "tank_excess": airplane["balance"]["tank_excess"], # 14
        "mlg_xcg_margin": airplane["inputs"]["x_mlg"] - airplane["balance"]["xcg_aft"], # 15
        "d_TO": d_TO # 16
    }
    return restricoes

def norm(x, lb, ub):
    return (np.array(x) - lb) / (ub - lb)

def denorm(x_norm, lb, ub):
    x = lb + np.array(x_norm) * (ub - lb)
    return np.clip(x, lb, ub)

def fun(airplane_name, x_norm):
    x = denorm(x_norm, lb, ub)
    dados = calcular_restricoes(criar_airplane_parametros(airplane_name, x))
    return 1e10 if dados is None else dados["W0"] / W0

def avaliar_restricoes(airplane_name, x_norm):
    x = denorm(x_norm, lb, ub)
    airplane = criar_airplane_parametros(airplane_name,x)
    dados = calcular_restricoes(criar_airplane_parametros(airplane_name, x))
    if dados is None:
        # Se quebrou, retorna valores muito fora de 1.0
        return np.array([100.0 if op == ">" else -100.0 for _, op, *_ in RESTRICOES])
        
    # NORMALIZAÇÃO DAS RESTRIÇÕES: Divide pelo limite para igualar a escala
    return np.array([dados[key] / (abs(limite) if limite != 0 else 1.0) for key, _, limite, _, _ in RESTRICOES])

def otimizar_aviao(airplane_name, parametros, x0, restricoes) -> tuple[Any, Any, Any, dict[Any, Any], float]:
  lb = np.array([limite[0] for limite in parametros.values()])
  ub = np.array([limite[1] for limite in parametros.values()])
  try:
    start = time.perf_counter()
    result = scipy.optimize.minimize(
        fun=fun,
        x0=norm(x0, lb, ub),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * len(parametros),
        constraints=constraints,
        options={"ftol": 1e-7, "disp": True, "maxiter": 500, "eps": 1e-3}, 
    )
    result.elapsed = time.perf_counter() - start
    result.x = denorm(result.x, lb, ub)
    result.airplane = criar_airplane_parametros(airplane_name, result.x)
    result.restricoes = calcular_restricoes(result.airplane)
    if result.restricoes is None:
        raise ValueError("Erro nas restrições.")
    return result
  except Exception:
     raise ValueError("Erro na otimização.")

In [ ]:
print("\n=== 2. Default Aircraft Optimization ===")
airplane_name = "fokker100"

parametros = {
    "AR_w": [7, 12],
    "S_w": [80, 120]
}

restricoes = {
    "b_w": [None, 30]
}

print("""
Minimizar W0(x)
w.r.t. x = [AR_w, S_w]
x0 = [7.5, 90]
Restrições:
[7, 80] <= [AR_w, S_w] <= [12, 120]
b_w(x) - 30 <= 0
""")

print(result)
print("\n1. Fill the following table with the optimization results:")
# rows: starting point, optimized point
# columns: AR_w, S_w, MTOW, b_w
# 7.5, 90m^2

print("\n  -> LATEX: ")

print("\n2. What is the relative improvement of the objective function, in percentage, after the optimization?")

print("\n3. How many function calls were needed in this optimization?")
print(f"Número de iterações: {result.nit}")
print(f"Número de chamadas da função: {result.nfev}")
print(f"Número de cálculos do Jacobiano: {result.njev}")
print("\n4. Is the optimum constrained? What are the active constraints?")


In [ ]:
print("\n=== 3. Team Aircraft Optimization ===")
airplane_name = "Tomav"

parametros = { # parâmetro: (mínimo, máximo)
    # Asa
    "delta_xr_w": (-8, 0),
    "S_w": (250, 400),
    "sweep_w": (25, 45),
    "AR_w": (6, 10.5),
    "taper_w": (0.17, 1),
    "tcr_w": (0.11, 0.175),
    "tct_w": (0.08, 0.12),
    "c_flap_c_wing": (0.15, 0.33),
    "b_flap_b_wing": (0.40, 0.65),
    "c_slat_c_wing": (0, 0.17),
    "b_slat_b_wing": (0, 0.90),
    "x_tank_c_w": (0.1, 0.7),
    
    # Empenagem Horizontal
    "Cht": (0.85, 1.2),
    "AR_h": (3.4, 6),
    "sweep_h": (20, 45),
    "taper_h": (0.35, 0.60),
    "tcr_h": (0.09, 0.12),
    "tct_h": (0.08, 0.12),
    
    # Empenagem Vertical
    "Cvt": (0.075, 0.11),
    "AR_v": (1.2, 2),
    "sweep_v": (30, 50),
    "taper_v": (0.35, 0.60),
    "tcr_v": (0.095, 0.12),
    "tct_v": (0.08, 0.12),
}

restricoes = { # restrição: (mínimo, máximo), None para ignorar
  "deltas_wlan": (0, None), # 1
  "SM_fwd": (None, 0.30), # 2
  "SM_aft": (0.05, None), # 3
  "frac_nlg_aft": (0.03, None), # 4
  "frac_nlg_fwd": (None, 0.18), # 5
  "alpha_tipback": (15, None), # 6
  "alpha_tailstrike": (10, None), # 7
  "phi_overturn": (None, 63), # 8
  # Restrição 9 já está definida no input b_tank_b_w = 0.95
  "b_w": (52, 62), # 10
  # Restrição 11 já está definida no input y_mlg = 5.5
  "tail_height": (None, 20), # 12
  # Restrições adicionais:
  "CLv": (None, 0.75), # 13
  "tank_excess": (0, None), # 14
  "mlg_xcg_margin": (0, None), # 15
  "d_TO": (None, 2900) # 16
}

airplane_original = standard_airplane(airplane_name)

print("\n1. Give the optimization problem definition and reasons behind the selection of design variables.")
print("""
Minimizar W0(x)
w.r.t. x = 6 variáveis
x0 = avião final PRJ-22
Restrições:
1. Landing requirement: deltaS_wlan ≥ 0
2. Static stability: SM_fwd ≤ 0.30
3. Static stability: SM_aft ≥ 0.05
4. Nose landing gear weight fraction: frac_nlg_fwd ≤ 0.18
5. Nose landing gear weight fraction: frac_nlg_aft ≥ 0.03
6. Main landing gear position: alpha_tipback ≥ 15 deg
7. Main landing gear position: alpha_tailstrike ≥ 10 deg
8. Main landing gear position: phi_overturn ≤ 63 deg
9. Fuel tank volume: b_tank_b_w ≤ 0.95 (já está definido b_tank_b_w = 0.95)
10. Wingspan limit for ICAO gate constraint: see Tab. 1
  - Wingspan = b_w = 58.86425061104575m -> Code E -> 52m < b_w < 65m
11. Wheel span limit for ICAO gate constraint: see Tab. 1
  - Code E -> 9 < 2*y_mlg < 14
  - 9 < 2*5.5 < 14 -> OK (não entra como restrição)
12. Heihgt limit for FAA Airplane Design Group: see Tab. 2
- Wingspan = 58.86425061104575m -> ADG V
- Tail Height = zt_v - z_lg = 12.507446283245763m - (-5.585m) = 18,09244628m -> ADG IV
- Pior: ADG V -> Tail Height = zt_v - z_lg < 20m
""")

print("\n2. Explain if new constraints or objectives were added.")
print("""
Restrições adicionais:
13. CLv <= 0.75 (controle lateral)
14. tank_excess >= 0 (combustível suficiente)
15. x_mlg >= x_cg_aft (não tombar)
16. d_TO < 2900 (decolar)
""")

print("\n3. Generate a table comparing values of design variables, constraints, and objective of the initial and the optimized configurations.")

print("\n4. Which optimizer have you used? Why?")
print("scipy.optimize.minimize com SLSQP, porque usei antes")

print("\n5. What is the relative improvement of the objective function, in percentage, after the optimization?")
print("\n6. How many function calls were needed in this optimization? How long did it take (seconds)?")
print(f"Número de iterações: {result.nit}")
print(f"Número de chamadas da função: {result.nfev}")
print(f"Número de cálculos do Jacobiano: {result.njev}")
print(f"Tempo de otimização: {elapsed:.4f} seconds")

print("\n7. Generate charts with the optimization history of design variables, constraints, and objective.")
print("\n8. Is the optimum constrained? What are the active constraints?")


ATTENTION: It is really important to normalize input variables and constraints. For example, the aft static margin constraint can be defined as: (SM_aft/0.05 −1) ≥ 0.

In [ ]:
print("\n=== Salvando avião ===")
with open("Lab2/tomav_atual.json", "w") as f:
    json.dump(???, f, indent=4)
print("\nDicionário do avião salvo em toanmav.json.")

with open("Lab2/tomav_otimizado.json", "w") as f:
    json.dump(???, f, indent=4)
print("\nDicionário do avião salvo em tomav.json.")

In [ ]:
print("\n=== 4. Multiobjective Optimization ===")
airplane_name = "Tomav"

print("""
Minimizar W0(x) e Wf(x)
w.r.t. x = 6 variáveis
x0 = avião final PRJ-22
Restrições: iguais
Use MOGA algorithm from pymoo.
""")

print("\n2. Plot the Pareto Front.")
print("\n3. Show the number of generations and individuals per generation.")
print("\n4. How would you use the results from Sec. 3 to verify the convergence of this multiobjective optimization?")
print("\n5. Choose at least three aircraft from distinct regions of the Pareto Front. Draw their planforms and discuss how their differences impact the objective function values.")

In [ ]:
plt.show()